# Agent: classify

Develop and test **`agentic_scd.agents.classify.classify_node`** in isolation.

## What this agent does

```mermaid
flowchart LR
    U["new_signals<br/>DisruptionSignal[]"]:::faded --> A1
    subgraph A["classify_node (per signal)"]
        A1["lowercase title + raw_text"] --> A2["count CATEGORY_KEYWORDS hits"]
        A2 --> A3{"any hits?"}
        A3 -->|yes| A4["pick best category"]
        A3 -->|no| A5["category = 'other'"]
        A4 --> A6["risk = f(hits, source_reliability)"]
        A5 --> A6
    end
    A6 --> D["classifications<br/>Classification[]"]
    D --> DOWN["downstream: impact · forecast · simulate · recommend"]:::faded
    classDef faded fill:#eee,stroke:#bbb,color:#888;
```

**State contract**

- **Reads:** `new_signals` (list of `DisruptionSignal`)
- **Writes:** `classifications` (list of `Classification`: category, risk_score, rationale)
- **Fallback / degradation:** no keyword hits → category `'other'`; missing `source_reliability` → 0.5

**Phase 3** replaces this rule/keyword stub with Groq classification/extraction + a fine-tuned DistilBERT risk score, behind the same `classify_node` signature.

## Is the DB up? (optional)

In [ ]:
# Optional: this agent runs fine offline on synthetic sample state. This snippet just
# reports whether the live DB is reachable (Setup section of 00_orchestration brings
# it up).
from agentic_scd.devtools import db_status

status = db_status()
print(status.detail)
if not status:
    print("Proceeding offline with synthetic sample state — fine for iterating here.")

## Build a representative input state

In [ ]:
from agentic_scd.devtools import sample_state

state = sample_state(count=2)
print("Input new_signals:")
for s in state["new_signals"]:
    print(" -", s.title)

## Call `classify_node` in isolation

In [ ]:
from agentic_scd.agents.classify import classify_node, classify_signal

state.update(classify_node(state))
print("Output classifications:")
for c in state["classifications"]:
    print(f" - {c.category} (risk {c.risk_score:.2f}): {c.rationale}")

# Or classify a single signal directly while iterating:
one = classify_signal(state["new_signals"][0])
print("\nsingle:", one)

## Iterate here

This is your dev surface: tweak the input above, re-run, and watch `classify_node`'s output change. When you deepen this agent in its phase, keep the node signature the same so the rest of the graph is unaffected.